In [34]:
from data_preprocessing import create_train_test_val_sets, get_processed_df

#Create test train splits
x_mendeley, y_mendeley = get_processed_df(r"..\data\raw\Mendeley Dataset.csv")
x_kaggle, y_kaggle= get_processed_df(r"..\data\raw\dataset_phishing.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
kaggle_sets = create_train_test_val_sets(x_kaggle,y_kaggle, label_col="Label", test_size=0.2, n_splits=5)

----------Processing None Dataset----------

Class Distribution:
Label
0    0.518415
1    0.481585
Name: proportion, dtype: float64
int64

Total Missing Values: 0
No categorical features to hash
Shape After Processing: (247950, 42)
True
----------Processing None Dataset----------

Class Distribution:
Label
legitimate    0.5
phishing      0.5
Name: proportion, dtype: float64
object

Total Missing Values: 0
Shape After Processing: (11430, 32856)
True
Train/validation/test split prepared: 210757 instances for training, 37193 instances for validation, 49590 instances for testing
Stratified 5-fold CV splits created.
Train/validation/test split prepared: 9715 instances for training, 1715 instances for validation, 2286 instances for testing
Stratified 5-fold CV splits created.


In [35]:
import joblib 

#import phase 1 models
xgb_mendeley = joblib.load('./models/phase_1/xgboost_mendeley_no_fs.joblib')
xgb_kaggle = joblib.load('./models/phase_1/xgboost_kaggle_no_fs.joblib')
logreg_mendeley = joblib.load('./models/phase_1/logreg_mendeley_no_fs.joblib')
logreg_kaggle = joblib.load('./models/phase_1/logreg_kaggle_no_fs.joblib')
rf_mendeley = joblib.load('./models/phase_1/rf_mendeley_no_fs.joblib')
rf_kaggle = joblib.load('./models/phase_1/rf_kaggle_no_fs.joblib')

In [ ]:
from sklearn.feature_selection import mutual_info_classif, f_classif
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp

def score_feature_redundancy(dataset, dataset_name, selected_features):
    """
    Calculates the statistical relevance to the target variable
    """
    X_selected = dataset['x_train'][selected_features]
    y = dataset['y_train']

    #use chi2 because more optimized for sprase matrices
    if 'kaggle' in dataset_name.lower():      
        X_sparse = sp.csr_matrix(X_selected.values)
        
        #calculate relevance score 
        relevance_scores, _ = f_classif(X_sparse, y)
        relevance_scores = np.nan_to_num(relevance_scores, nan=0.0)
        
        #calculate redundancy using cosine similarity
        similarity_matrix = cosine_similarity(X_sparse.T, dense_output=False)
        if sp.issparse(similarity_matrix):
            similarity_matrix.setdiag(0)
        else:
            np.fill_diagonal(similarity_matrix, 0)
    else:        
        #calculate relevance score 
        relevance_scores = mutual_info_classif(X_selected, y, discrete_features='auto', random_state=42)
        
        #calculate redundancy using pearson correlation
        similarity_matrix = X_selected.corr().abs().to_numpy(copy=True)
        np.fill_diagonal(similarity_matrix, 0)

    #normalize scores
    relevance_normalized = pd.Series(relevance_scores).rank(pct=True).values

    feature_scores = pd.DataFrame({
        'Feature': X_selected.columns,
        'Matrix_Index': np.arange(len(X_selected.columns)), 
        'Relevance_to_Target': np.asarray(relevance_normalized).flatten(),
    })

    return feature_scores, similarity_matrix

def remove_redundant_features(dataset, dataset_name, selected_features, stability_dict=None, max_redundancy=0.85, min_score=0.01):
    """
    Calculates a unified 'Priority Score' and uses a greedy selection loop to keep top-performing features while discarding highly 
    correlated features. Hard fake indicators are granted immunity 
       and will never be dropped, regardless of redundancy or score thresholds.
    2. Evasion Weights: Features that are cheap for a hacker to manipulate (e.g., URL length) 
       receive a priority penalty, while expensive structural features receive a priority boost.
    """
    print(f'Initial feature counts {len(selected_features)}')

    #certain features are harder to fake than others, so we don't want to drop them no matter what
    if 'kaggle' in dataset_name.lower():
        important_features = [
            'domain_age', 'whois_registered_domain', 'google_index', 'page_rank', 
            'web_traffic', 'https_token', 'nb_at', 'nb_hyphens', 'nb_dslash'
        ]
    else:
        important_features = ['entropy_of_domain', 'entropy_of_url', 'number_of_subdomains', 'number_of_at_in_url', 'number_of_hyphens_in_url', 'number_of_slash_in_url']

    scores_df, similarity_matrix = score_feature_redundancy(dataset, dataset_name, selected_features)
    
    #some features are harder to fake - so they should be kept if they are both redundant
    scores_df['Ability_to_Fake_Weight'] = 1.0 
    
    if 'kaggle' in dataset_name.lower():
        #easy to change between attacks --> weigh less
        cheap_url_features = ['length_url', 'length_hostname', 'length_words_raw', 'nb_www', 'nb_com']
        scores_df.loc[scores_df['Feature'].isin(cheap_url_features), 'Ability_to_Fake_Weight'] = 0.8
        
        #harder to change and fake --> weigh more 
        expensive_url_features = ['ip', 'ratio_digits_url', 'ratio_digits_host', 'abnormal_subdomain']
        scores_df.loc[scores_df['Feature'].isin(expensive_url_features), 'Ability_to_Fake_Weight'] = 1.3
                
    elif 'mendeley' in dataset_name.lower():
        #easy to change between attacks --> weigh less
        cheap_url_features = ['url_length', 'domain_length', 'path_length']
        scores_df.loc[scores_df['Feature'].isin(cheap_url_features), 'Ability_to_Fake_Weight'] = 0.8
        
        #harder to change and fake --> weigh more 
        expensive_url_features = ['average_subdomain_length', 'entropy_of_url', 'entropy_of_domain']
        scores_df.loc[scores_df['Feature'].isin(expensive_url_features), 'Ability_to_Fake_Weight'] = 1.3

    if stability_dict is not None:
        scores_df['Stability_Score'] = scores_df['Feature'].map(stability_dict).fillna(0)
        scores_df['Priority_Score'] = (scores_df['Stability_Score'] + scores_df['Relevance_to_Target']) * scores_df['Ability_to_Fake_Weight']
    else:
        scores_df['Priority_Score'] = scores_df['Relevance_to_Target'] * scores_df['Ability_to_Fake_Weight']

    # Sort from best to worst
    scores_df = scores_df.sort_values(by='Priority_Score', ascending=False).reset_index(drop=True)

    #greedily choose features 
    final_feature_list = []
    dropped_indices = set()

    for _, row in scores_df.iterrows():
        feat = row['Feature']
        idx = row['Matrix_Index']
        is_important = feat in important_features

        #if important feature, keep it no matter what
        if idx in dropped_indices and not is_important:
            continue
        if row['Priority_Score'] < min_score and not is_important:
            continue
        final_feature_list.append(feat)

        #find correlated features to the chosen features
        if sp.issparse(similarity_matrix):
            correlations = similarity_matrix.getrow(idx).toarray().ravel()
        else:
            correlations = similarity_matrix[idx, :]
            
        redundant_idx = np.where(correlations > max_redundancy)[0]
        dropped_indices.update(redundant_idx)
        
    print(f"Removed {len(selected_features) - len(final_feature_list)} features.")
    print(f"Final feature count for: {len(final_feature_list)}\n")
    
    return final_feature_list, scores_df



In [37]:
#Evaluate and save 
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression


def evaluate_model(model_name, dataset, model, selected_features):
    
    X_train = pd.concat([dataset["x_train"], dataset["x_val"]]).reindex(columns=selected_features)
    X_test = dataset["x_test"].reindex(columns=selected_features)
    y_train = pd.concat([dataset["y_train"], dataset["y_val"]])
    y_test = dataset["y_test"]

    if model_name == 'LogReg':
        params = model.named_steps["model"].get_params()
        print(params)
        model_clone = Pipeline([
            ("scaler", MaxAbsScaler()),
            ("model", LogisticRegression(**params))
        ])
    else:
        model_clone = clone(model)
    model_clone.fit(X_train, y_train)

    y_pred = model_clone.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    return f1, prec, rec

In [38]:
import os 

#Run Mendeley
models_mendeley = {
    "XGBoost": xgb_mendeley,
    "RF": rf_mendeley,
    "LogReg": logreg_mendeley
}

selected_features, scores_df = remove_redundant_features(
    dataset=mendeley_sets,
    dataset_name="Mendeley",
    selected_features=mendeley_sets['x_train'].columns
)

final_results_mendeley = []

for model_name, model in models_mendeley.items():
    print(f"\nEvaluating {model_name}")

    f1, prec, rec = evaluate_model(model_name, mendeley_sets, model, selected_features)

    print(f"{model_name} | F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

    final_results_mendeley.append({
        "dataset": "Mendeley",
        "model": model_name,
        "num_features": len(selected_features),
        "f1": f1,
        "precision": prec,
        "recall": rec,
        "features": selected_features
    })

os.makedirs(f"../results/phase_3/redundancy_reports", exist_ok=True)
final_results_mendeley_df = pd.DataFrame(final_results_mendeley)
final_results_mendeley_df.to_csv(f"../results/phase_3/redundancy_reports/mendeley_redundancy_scores.csv", index=False)

Initial feature counts 41
Removed 5 features.
Final feature count for: 36


Evaluating XGBoost
XGBoost | F1: 0.9678, Precision: 0.9786, Recall: 0.9572

Evaluating RF
RF | F1: 0.9987, Precision: 0.9982, Recall: 0.9992

Evaluating LogReg
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}
LogReg | F1: 0.7838, Precision: 0.8309, Recall: 0.7418


In [39]:
#Run Kaggle
models_kaggle = {
    "XGBoost": xgb_kaggle,
    "RF": rf_kaggle,
    "LogReg": logreg_kaggle
}

selected_features, scores_df = remove_redundant_features(
    kaggle_sets,
    "Kaggle",
    kaggle_sets['x_train'].columns,
)

#Evaluate and save kaggle
final_results_kaggle = []

for model_name, model in models_kaggle.items():
    print(f"\nEvaluating {model_name}")
    f1, prec, rec = evaluate_model(model_name, kaggle_sets, model, selected_features)

    print(f"{model_name} | F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

    final_results_kaggle.append({
        "dataset": "Kaggle",
        "model": model_name,
        "num_features": len(selected_features),
        "f1": f1,
        "precision": prec,
        "recall": rec,
        "features": selected_features
    })

final_results_kaggle_df = pd.DataFrame(final_results_kaggle)
final_results_kaggle_df.to_csv(f"../results/phase_3/redundancy_reports/kaggle_redundancy_scores.csv", index=False)

Initial feature counts 32855


C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [0 0 0 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Removed 840 features.
Final feature count for: 32015


Evaluating XGBoost


C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")


XGBoost | F1: 1.0000, Precision: 1.0000, Recall: 1.0000

Evaluating RF
RF | F1: 1.0000, Precision: 1.0000, Recall: 1.0000

Evaluating LogReg
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}
LogReg | F1: 1.0000, Precision: 1.0000, Recall: 1.0000


In [40]:
import ast 

#XGBoost + Mendeley
def save_stability_redundancy(file_name, model_name, model, dataset_name, dataset, score_threshold=0.3, min_composite_score=0.20, max_redundancy=0.85):
    stability_df = pd.read_csv(file_name)
    stability_df["Stable features"] = stability_df["Stable features"].apply(ast.literal_eval)
    final_results = []

    for _, row in stability_df.iterrows():
        method = row["FS_Method"]
        stable_features_dict = row["Stable features"]

        print(f"\nEvaluating Composite Score for {model_name} - {method}")
        feature_names = list(stable_features_dict.keys())

        final_features, scores_df = remove_redundant_features(
            dataset=dataset, 
            dataset_name=dataset_name, 
            selected_features=feature_names, 
            stability_dict=stable_features_dict,
            max_redundancy=max_redundancy,
            min_score=min_composite_score
        )

        print(f"Selected {len(final_features)} out of {len(feature_names)} features.")

        # 5. Evaluate the model using only the best features
        f1, prec, rec = evaluate_model(model_name, dataset, model, final_features)

        final_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "FS_Method": method,
            "Stability_F1": row["Avg_Val_F1"],
            "Num_Stable_Features": len(stable_features_dict),
            "Num_Final_Features": len(final_features),
            "F1": f1,
            "Precision": prec,
            "Recall": rec,
            "Composite_Threshold": min_composite_score,
            "Final_Features": final_features
        })

        # Optional: Save the detailed feature scoreboard for this specific run for your report
        os.makedirs(f"../results/phase_3/detailed_feature_scores", exist_ok=True)
        scores_df.to_csv(f"../results/phase_3/detailed_feature_scores/{dataset_name}_{model_name}_{method}_scores.csv", index=False)

    # Save the final aggregated report
    os.makedirs("../results/phase_3/final_reports", exist_ok=True)
    results_df = pd.DataFrame(final_results)
    results_df.to_csv(f"../results/phase_3/final_reports/final_results_{dataset_name.lower()}_{model_name.lower()}.csv", index=False)
    

#Mendeley
os.makedirs("../results/phase_3/final_reports", exist_ok=True)
save_stability_redundancy("../results/phase_3/stability_reports/XGBoost_Mendeley_stability_report.csv", 'XGBoost', xgb_mendeley, 'Mendeley', mendeley_sets)
save_stability_redundancy("../results/phase_3/stability_reports/Random Forest_Mendeley_stability_report.csv", 'RF', rf_mendeley, 'Mendeley', mendeley_sets)
save_stability_redundancy("../results/phase_3/stability_reports/Logistic Regression_Mendeley_stability_report.csv", 'LogReg', logreg_mendeley, 'Mendeley', mendeley_sets)

#Kaggle
save_stability_redundancy("../results/phase_3/stability_reports/XGBoost_Kaggle_stability_report.csv", 'XGBoost', xgb_kaggle, 'Kaggle', kaggle_sets)
save_stability_redundancy("../results/phase_3/stability_reports/Random Forest_Kaggle_stability_report.csv", 'RF', rf_kaggle, 'Kaggle', kaggle_sets)
save_stability_redundancy("../results/phase_3/stability_reports/Logistic Regression_Kaggle_stability_report.csv", 'LogReg', logreg_kaggle, 'Kaggle', kaggle_sets)


Evaluating Composite Score for XGBoost - anova
Initial feature counts 30
Removed 5 features.
Final feature count for: 25

Selected 25 out of 30 features.

Evaluating Composite Score for XGBoost - chi2
Initial feature counts 25
Removed 5 features.
Final feature count for: 20

Selected 20 out of 25 features.

Evaluating Composite Score for XGBoost - mi
Initial feature counts 24
Removed 4 features.
Final feature count for: 20

Selected 20 out of 24 features.

Evaluating Composite Score for XGBoost - variance
Initial feature counts 27
Removed 4 features.
Final feature count for: 23

Selected 23 out of 27 features.

Evaluating Composite Score for RF - anova
Initial feature counts 25
Removed 4 features.
Final feature count for: 21

Selected 21 out of 25 features.

Evaluating Composite Score for RF - chi2
Initial feature counts 20
Removed 4 features.
Final feature count for: 16

Selected 16 out of 20 features.

Evaluating Composite Score for RF - mi
Initial feature counts 25
Removed 5 featur

C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")



Evaluating Composite Score for XGBoost - chi2
Initial feature counts 29058


C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [0 0 0 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Removed 497 features.
Final feature count for: 28561

Selected 28561 out of 29058 features.


...


Evaluating Composite Score for RF - anova
Initial feature counts 22083
Removed 489 features.
Final feature count for: 21594

Selected 21594 out of 22083 features.

Evaluating Composite Score for RF - chi2
Initial feature counts 22206


C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [0 0 0 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Removed 490 features.
Final feature count for: 21716

Selected 21716 out of 22206 features.

Evaluating Composite Score for RF - mi
Initial feature counts 29490
Removed 511 features.
Final feature count for: 28979

Selected 28979 out of 29490 features.

Evaluating Composite Score for RF - variance
Initial feature counts 57
Removed 9 features.
Final feature count for: 48

Selected 48 out of 57 features.

Evaluating Composite Score for LogReg - anova
Initial feature counts 22083
Removed 489 features.
Final feature count for: 21594

Selected 21594 out of 22083 features.
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}

Evaluating Composite Score for LogReg - chi2
Initial feature counts 29058


C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [0 0 0 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Removed 497 features.
Final feature count for: 28561

Selected 28561 out of 29058 features.
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}

Evaluating Composite Score for LogReg - mi
Initial feature counts 24016
Removed 511 features.
Final feature count for: 23505

Selected 23505 out of 24016 features.
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}

Evaluating Composite Score for LogReg - variance
Initial feature counts 57
Removed 9 features.
Final feature count for: 48

Selected 48 out of 57 features.
{'C': 10, 'class

In [ ]:
#Union the results 

for file in os.listdir("../results/phase_3/final_reports/"):
    df = pd.read_csv(f"../results/phase_3/final_reports/{file}")
    set_feature_list = []
    for feature_list in df["Final_Features"]:
        set_feature_list.append(ast.literal_eval(feature_list))
    
    union = list(set.union(*map(set, set_feature_list)))

    if df['Dataset'][0] == "Kaggle":
        dataset = kaggle_sets
        if df['Model'][0] == 'XGBoost':
            model = xgb_kaggle
        elif df['Model'][0] == 'LogReg':
            model = logreg_kaggle
        else:
            model = rf_kaggle
    else:
        dataset = mendeley_sets
        if df['Model'][0] == 'XGBoost':
            model = xgb_mendeley
        elif df['Model'][0] == 'LogReg':
            model = logreg_mendeley
        else:
            model = rf_mendeley

    f1, prec, rec = evaluate_model(df['Model'][0], dataset, model, union)
    print(f"{df['Model'][0]} | {df['Dataset'][0]} | F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")
    
    union_df = pd.DataFrame([{
        "Dataset": df["Dataset"][0],
        "Model": df["Model"][0],
        "F1": f1,
        "Precision": prec,
        "Recall": rec,
        "Final_Feature_Union": union
    }])

    os.makedirs("../results/final_results/", exist_ok=True)
    union_df.to_csv(f'../results/final_results/{file}')



{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}
LogReg | Kaggle | F1: 1.0000, Precision: 1.0000, Recall: 1.0000
RF | Kaggle | F1: 1.0000, Precision: 1.0000, Recall: 1.0000


C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
C:\Users\nive0\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")


XGBoost | Kaggle | F1: 1.0000, Precision: 1.0000, Recall: 1.0000
{'C': 10, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 5000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'saga', 'tol': 0.001, 'verbose': 0, 'warm_start': False}
LogReg | Mendeley | F1: 0.7827, Precision: 0.8297, Recall: 0.7407
RF | Mendeley | F1: 0.9986, Precision: 0.9982, Recall: 0.9991
XGBoost | Mendeley | F1: 0.9689, Precision: 0.9791, Recall: 0.9590
